# Telco Customer Churn Analysis

I'm starting by loading the raw Telco customer churn dataset to inspect customer demographics, subscribed services, tenure, and billing details.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

raw = pd.read_csv("../raw_unedited/WA_Fn-UseC_-Telco-Customer-Churn.csv", dtype={"TotalCharges": "string"})
raw.head()

Next I'm checking for duplicate rows before anything else touches this data. I'm also verifying that customerID uniquely identifies each record.

In [ ]:
d = raw.copy()
for c in d.select_dtypes(include=["object", "string"]).columns:
    d[c] = d[c].astype("string").str.strip()
    d[c] = d[c].replace({"": pd.NA})

assert d["customerID"].is_unique, "customerID must be unique"
print(f"Verified {len(d):,} unique customers with 0 duplicate rows.")

Now I'm inspecting the missing values in TotalCharges. There are 11 records where TotalCharges is blank. Looking closely, these all correspond to new customers with tenure = 0 who haven't had their first billing cycle yet. I'm converting these to NaN/null rather than imputing, because retaining the true missingness reflects the reality of new accounts.

In [ ]:
blank_count = d["TotalCharges"].isna().sum()
d["TotalCharges"] = pd.to_numeric(d["TotalCharges"], errors="coerce")
print(f"TotalCharges blank count: {blank_count}")
print("Tenure distribution for customers with missing TotalCharges:")
print(d[d["TotalCharges"].isna()]["tenure"].value_counts())

Next, I'll calculate the baseline churn rate across the entire customer base to understand overall customer retention.

In [ ]:
churn_counts = d["Churn"].value_counts()
churn_rate = (d["Churn"] == "Yes").mean()
print(f"Total Customers: {len(d):,}")
print(f"Churned Customers: {churn_counts.get('Yes', 0):,} ({churn_rate:.2%})")

Here I'm analyzing churn rate across contract types. Month-to-month contracts have dramatically higher churn compared to one-year and two-year commitments.

In [ ]:
contract_summary = d.groupby("Contract").agg(
    customers=("customerID", "count"),
    churned=("Churn", lambda x: (x == "Yes").sum()),
    churn_rate=("Churn", lambda x: (x == "Yes").mean()),
    avg_monthly_charges=("MonthlyCharges", "mean")
).reset_index()

print(contract_summary)

plt.figure(figsize=(8, 5))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    data=contract_summary.sort_values("churn_rate", ascending=False),
    x="Contract",
    y="churn_rate",
    palette=["#d95f02", "#7570b3", "#1b9e77"]
)
plt.title("Churn Rate by Contract Type", fontsize=13, weight="bold")
plt.ylabel("Churn Rate")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1%}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', weight='bold')
plt.tight_layout()
plt.show()